In [1]:
from pathlib import Path
import json
import sys
import subprocess

This tutorial follows the Bindcraft example on https://github.com/martinpacesa/BindCraft. 
First we need to make sure all libraries are defined. Click on the hexagon icon on the far left of the bwVisu browser window and load the compiler/gnu/11.3 module by clicking on the Load button next to the entry. 

Execute the next cell to verify that the module is loaded. If compiler/gnu/11.3 is not in the list, click on the bindcraft kernel on the top right of the window to reload the kernel. Select bindcraft and try again!

In [2]:
# export paths for pyrosetta

! export LD_LIBRARY_PATH=./mnt/sds-hd/sd25g005/bindcraft/lib:$LD_LIBRARY_PATH
! echo $LD_LIBRARY_PATH

# test if modules and fortran libraries are loaded
! module list

/opt/bwhpc/common/compiler/gnu/11.3.0/lib64:/.singularity.d/libs

Currently Loaded Modules:
  1) system/singularity/3.11.3   2) compiler/gnu/11.3

 



First we need to define the working directory, and upload the PDL1.pdb file to the working directory.

In [ ]:
BINDCRAFT_WORKING_DIR = Path.cwd() / "protein_design_w_Bindcraft"

Now we need to define the input file. For a detailed explanation on the parameters, see the [Bindcraft github](https://github.com/martinpacesa/BindCraft?tab=readme-ov-file#running-the-script-locally-and-explanation-of-settings).

In [4]:
data = {
    "design_path": str(BINDCRAFT_WORKING_DIR),
    "binder_name": "PDL1",
    "starting_pdb": str(BINDCRAFT_WORKING_DIR / "PDL1.pdb"),
    "chains": "A",
    "target_hotspot_residues": "56",
    "lengths": [65, 150],
    "number_of_final_designs": 100,
}

bindcraft_settings = BINDCRAFT_WORKING_DIR / "input.json"

with open(bindcraft_settings, "w") as json_file:
    json.dump(data, json_file, indent=4)  # Use indent for pretty-printing

Now we need to link to the Bindcraft directory in the s25g005 Speichervorhaben. If you want to use other filters or advanced settings, choose them from the list of files.

In [5]:
BINDCRAFT_DIR = Path("/mnt/sds-hd/sd25g005/bindcraft/BindCraft")

BINDCRAFT_PY = BINDCRAFT_DIR / "bindcraft.py"
BINDCRAFT_FILTERS = BINDCRAFT_DIR / "settings_filters" / "default_filters.json"
BINDCRAFT_ADVANCED = (
    BINDCRAFT_DIR / "settings_advanced" / "default_4stage_multimer.json"
)

subprocess_command = [
    sys.executable,
    "-u",
    BINDCRAFT_PY,
    "--settings",
    bindcraft_settings,
    "--filters",
    BINDCRAFT_FILTERS,
    "--advanced",
    BINDCRAFT_ADVANCED,
]

Now we can start the Bindcraft run by executing the cells below.

In [ ]:
subprocess.run(subprocess_command)

Available GPUs:
NVIDIA A401: gpu
┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2026 [Rosetta PyRosetta4.conda.ubuntu.cxx11thread.serialization.Ubuntu.python310.Release 2026.03+release.5e498f1409c68ade56c8ce5842bf79e1b02e8db4 2026-01-13T13:24:11] retrieved from: http://www.pyrosetta.org
Running binder design for target input
Design settings used: default_4stage_multimer
Filter

You find the results in your `WORKING_DIR` on the right. Note that it takes quite a long time to complete the run, see the discussion [here](https://github.com/martinpacesa/BindCraft/issues/83) to get an idea.

## Known issues
Currently we have to deactivate the trajectory gifs via ffmpeg. See https://github.com/martinpacesa/BindCraft/issues/20 